In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    load_dyst_samples,
    plot_output_logits_across_layers,
)


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

### Load Data

In [ ]:
split_name = "gift-eval"

subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Lorenz"
    # ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    # Prepare input series
    sample_idx = 0
    selected_dim = 0
    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]

    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    # Load the path to gift-eval data, which is stored in .env file
    split_dir = os.path.join(WORK_DIR, DATA_DIR, split_name)
    # system_name = "electricity/D"
    system_name = "m4_monthly"

    to_univariate = False
    term = "long"

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    print("Dataset frequency: ", dataset.freq)
    print("Prediction length: ", dataset.prediction_length)
    print("Number of windows in the rolling evaluation: ", dataset.windows)

    test_split_iter = dataset.test_data
    test_data = next(iter(test_split_iter))

    test_input_split_iter = dataset.test_data.input

    input = next(iter(test_input_split_iter))
    print("Keys in the test data: ", input.keys())

    print("\n\nContext Item id: ", input["item_id"])
    print("Context Start Date: ", input["start"])
    print("Context Frequency: ", input["freq"])
    print(f"Context shape: {input['target'].shape}")

    test_label_split_iter = dataset.test_data.label
    label = next(iter(test_label_split_iter))
    print("\n\nForecast Item id: ", label["item_id"])
    print("Forecast Start Date: ", label["start"])
    print("Forecast Frequency: ", label["freq"])
    print(f"Forecast shape: {label['target'].shape}")

    fig, ax = plt.subplots(figsize=(5, 2))
    context = to_pandas(test_data[0])
    future_vals = to_pandas(test_data[1])
    prediction_length = future_vals.shape[0]
    print(f"prediction length: {prediction_length}")
    context.plot(color="black", linewidth=1, ax=ax)
    future_vals.plot(color="tab:blue", linewidth=1, ax=ax)
    ax.grid(which="both")
    # ax.legend("ground truth", loc="upper left")
    plt.show()
    num_nans_context = context.isna().sum()
    num_nans_future_vals = future_vals.isna().sum()

    context = torch.tensor(context).unsqueeze(0)
    future_vals = torch.tensor(future_vals).unsqueeze(0)
    print(f"num nans context: {num_nans_context}")
    print(f"num nans future vals: {num_nans_future_vals}")
    print(f"context length: {context.shape}")
    print(f"future vals shape: {future_vals.shape}")


### Load Pipeline

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
num_samples = 10
do_sample = True if num_samples > 1 else False

pipeline.remove_all_hooks()

print("Adding CA head attribution hooks")
pipeline.add_head_attribution_hooks(
    [(i, j) for i in range(num_layers) for j in range(num_heads)],
    attention_type="ca",
)

print("Adding SA head attribution hooks")
pipeline.add_head_attribution_hooks(
    [(i, j) for i in range(num_layers) for j in range(num_heads)],
    attention_type="sa",
)

print("Adding MLP attribution hooks")
pipeline.add_mlp_attribution_hooks([i for i in range(num_layers)])

print("Adding read stream hooks")
pipeline.add_read_stream_hooks([i for i in range(num_layers)])

print("Adding SA attn output attribution hooks")
pipeline.add_attn_attribution_hooks([i for i in range(num_layers)], attention_type="sa")
print("Adding CA attn output attribution hooks")
pipeline.add_attn_attribution_hooks([i for i in range(num_layers)], attention_type="ca");

In [ ]:
pred, encdec_out = pipeline.predict_with_full_outputs(  # type: ignore
    context=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    do_sample=do_sample,  # greedy decoding if num_samples == 1
    use_cache=True,
    return_dict_in_generate=True,
    output_scores=True,
    limit_prediction_length=False,
)

In [ ]:
do_sample

### Visualize

In [ ]:
# Prepare context and predictions
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]

fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(future_timesteps, future_vals_np, color="black", linewidth=1, linestyle="--", label="Ground Truth")
for i in range(len(preds)):
    ax.plot(future_timesteps, preds[i], color=DEFAULT_COLORS[i], linewidth=1, alpha=0.5)
ax.plot(future_timesteps, np.median(preds, axis=0), color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")

# Save plot
save_fname = (
    f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name}"
)
fig.tight_layout()

In [ ]:
sa_head_outputs = {
    i: [collect_attributions(pipeline.sa_head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

ca_head_outputs = {
    i: [collect_attributions(pipeline.ca_head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

mlp_outputs = {i: collect_attributions(pipeline.mlp_attribution_outputs[i]) for i in range(num_layers)}

read_stream_outputs = {i: collect_attributions(pipeline.read_stream_outputs[i]) for i in range(num_layers)}

In [ ]:
for name, outs in [
    ("ca_head", ca_head_outputs),
    ("sa_head", sa_head_outputs),
    ("mlp", mlp_outputs),
    ("read_stream", read_stream_outputs),
]:
    for out in outs.values():
        shape = out[0].shape if name.endswith("head") else out.shape
        assert shape[0] == num_samples, f"{name} shape[0] = {shape[0]} != num_samples = {num_samples}"

In [ ]:
len(ca_head_outputs[0])

In [ ]:
layer_idx = 0
head_idx = 0
sample_idx = 0
timestep_idx = 0
ca_head_outputs[layer_idx][head_idx][sample_idx][timestep_idx].shape

In [ ]:
ca_head_outputs_normalized = {
    layer: [head_output / torch.norm(head_output, dim=-1, keepdim=True) for head_output in ca_head_outputs[layer]]
    for layer in ca_head_outputs
}
sa_head_outputs_normalized = {
    layer: [head_output / torch.norm(head_output, dim=-1, keepdim=True) for head_output in sa_head_outputs[layer]]
    for layer in sa_head_outputs
}

In [ ]:
# Normalize head outputs
def normalize_outputs(outputs):
    return {layer: [h / torch.norm(h, dim=-1, keepdim=True) for h in outputs[layer]] for layer in outputs}


ca_head_outputs_normalized = normalize_outputs(ca_head_outputs)
sa_head_outputs_normalized = normalize_outputs(sa_head_outputs)

# Get dimensions
num_layers, num_heads = len(ca_head_outputs), len(ca_head_outputs[0])
num_samples, num_timesteps, d_model = ca_head_outputs[0][0].shape

# Calculate gramians
ca_gramians = torch.zeros(num_layers, num_samples, num_timesteps, num_heads, num_heads)
sa_gramians = torch.zeros(num_layers, num_samples, num_timesteps, num_heads, num_heads)

for layer in range(num_layers):
    for h1, h2 in [(i, j) for i in range(num_heads) for j in range(num_heads)]:
        ca_gramians[layer, :, :, h1, h2] = torch.einsum(
            "b t d, b t d -> b t", ca_head_outputs_normalized[layer][h1], ca_head_outputs_normalized[layer][h2]
        )
        sa_gramians[layer, :, :, h1, h2] = torch.einsum(
            "b t d, b t d -> b t", sa_head_outputs_normalized[layer][h1], sa_head_outputs_normalized[layer][h2]
        )

# Plot mean gramian for first layer
plt.imshow(torch.mean(ca_gramians, dim=(1, 2))[0], vmin=-1, vmax=1, cmap="RdBu")
plt.colorbar(label="Dot Product")
plt.show()

In [ ]:
# Compute SVD of gramians and analyze head interactions
ca_singular_values = torch.zeros(num_layers, num_samples, num_timesteps, num_heads)
sa_singular_values = torch.zeros(num_layers, num_samples, num_timesteps, num_heads)

for layer in range(num_layers):
    for sample in range(num_samples):
        for timestep in range(num_timesteps):
            ca_singular_values[layer, sample, timestep] = torch.linalg.svd(ca_gramians[layer, sample, timestep]).S
            sa_singular_values[layer, sample, timestep] = torch.linalg.svd(sa_gramians[layer, sample, timestep]).S

# Calculate entropic rank
ca_p_vals = torch.sqrt(ca_singular_values) / (torch.sqrt(ca_singular_values).sum(dim=-1, keepdim=True))
sa_p_vals = torch.sqrt(sa_singular_values) / (torch.sqrt(sa_singular_values).sum(dim=-1, keepdim=True))
ca_entropy = -torch.sum(ca_p_vals * torch.log(ca_p_vals), dim=-1)
sa_entropy = -torch.sum(sa_p_vals * torch.log(sa_p_vals), dim=-1)
ca_mean_entropy = torch.mean(ca_entropy, dim=(1, 2))
sa_mean_entropy = torch.mean(sa_entropy, dim=(1, 2))

# Plot entropic ranks
# plt.figure(figsize=(5, 5))
plt.figure(figsize=(4, 6))
plt.plot(torch.exp(ca_mean_entropy), label="CA Heads", linewidth=2)
plt.plot(torch.exp(sa_mean_entropy), label="SA Heads", linewidth=2)
plt.xlabel("Layer", fontweight="bold")
plt.xticks(range(num_layers))
plt.ylabel("Entropic Rank $\\exp\\left(-\\sum_{i=1}^{n} p_i \\log(p_i)\\right)$", fontweight="bold")
plt.title("Entropic Rank of Head Outputs", fontweight="bold")
plt.legend(frameon=True)
plt.ylim(8.5, 12.5)
# plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def plot_head_similarity_with_magnitude(head_outputs, output_type, save_path=None):
    """Plot head similarity and magnitude heatmaps for each layer."""
    n_layers = 12
    n_heads = 12
    fig, axs = plt.subplots(3, 4, figsize=(16, 12))
    axs = axs.flatten()

    for layer_idx in range(n_layers):
        heads_data = head_outputs[layer_idx]
        similarities = np.zeros((n_heads, n_heads))
        magnitudes = np.zeros(n_heads)

        # Calculate magnitudes
        for i in range(n_heads):
            head_output = heads_data[i][0]
            magnitudes[i] = torch.norm(head_output, dim=-1).mean().item()

        # Calculate similarities
        for i in range(n_heads):
            for j in range(n_heads):
                i_flat = heads_data[i][0].reshape(-1, heads_data[i][0].shape[-1])
                j_flat = heads_data[j][0].reshape(-1, heads_data[j][0].shape[-1])
                similarities[i, j] = torch.nn.functional.cosine_similarity(i_flat, j_flat, dim=-1).mean().item()

        # Plot
        ax = axs[layer_idx]
        im = ax.imshow(similarities, cmap="RdBu", vmin=-1, vmax=1)
        ax.set_title(f"Layer {layer_idx}")
        ax.set_xlabel("Head Index")
        ax.set_ylabel("Head Index")

        # Add magnitude bars
        divider = make_axes_locatable(ax)
        ax_mag = divider.append_axes("right", size="15%", pad=0.1)
        ax_mag.barh(np.arange(n_heads), magnitudes, color="coral")
        ax_mag.set_ylim(ax_mag.get_ylim()[::-1])
        ax_mag.set_title("Magnitude")
        ax_mag.set_yticks([])

    plt.tight_layout()
    fig.colorbar(im, ax=axs, label="Cosine Similarity", shrink=0.6)
    fig.suptitle(f"{output_type.upper()} Head Output Similarity and Magnitude", fontsize=16)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


# Plot both CA and SA head outputs
plot_save_dir = "../figs/head_outputs"
fig_ca = plot_head_similarity_with_magnitude(ca_head_outputs, "ca", f"{plot_save_dir}/ca_head_similarity.png")
fig_sa = plot_head_similarity_with_magnitude(sa_head_outputs, "sa", f"{plot_save_dir}/sa_head_similarity.png")
plt.show()

In [ ]:
head_outputs = {
    i: [collect_attributions(pipeline.ca_head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

mlp_outputs = {i: collect_attributions(pipeline.mlp_attribution_outputs[i]) for i in range(num_layers)}

read_stream_outputs = {i: collect_attributions(pipeline.read_stream_outputs[i]) for i in range(num_layers)}

In [ ]:
for name, outs in [
    ("head", head_outputs),
    ("mlp", mlp_outputs),
    ("read_stream", read_stream_outputs),
]:
    for out in outs.values():
        shape = out[0].shape if name == "head" else out.shape
        assert shape[0] == num_samples, f"{name} shape[0] = {shape[0]} != num_samples = {num_samples}"


In [ ]:
layer_idx = 10
plot_output_logits_across_layers(
    pipeline,
    head_outputs[layer_idx],
    prediction_length,
    output_type="Head",
    title=f"CA Head Outputs for Layer {layer_idx}",
    plot_save_path=os.path.join(plot_save_dir, f"head_outputs_across_layers_{system_name}_layer{layer_idx}.pdf"),
)

### Pivot-head grouping and manipulation (CA + SA)
This section computes head groups, selects pivot heads, derives scaling multipliers from a similarity matrix and magnitudes, and applies pivot-head manipulation hooks for both cross-attention (CA) and self-attention (SA) across all layers.

Run these cells end-to-end to replace any earlier single-layer or CA-only setup.

In [ ]:
from pprint import pprint

import torch
import torch.nn.functional as F

EPSILON = 0.6  # similarity threshold

num_layers = int(pipeline.model.model.config.num_decoder_layers)
num_heads = int(pipeline.model.model.config.num_heads)


# Helper to compute similarity and magnitudes for all layers for a given attention outputs dict
# outputs[layer] is a list length num_heads of tensors shaped (B, T, d)
def compute_similarity_and_magnitudes(outputs: dict[int, list[torch.Tensor]]):
    layer_similarities: dict[int, torch.Tensor] = {}
    layer_magnitudes: dict[int, torch.Tensor] = {}
    for layer_idx in range(num_layers):
        heads = outputs[layer_idx]
        mags = torch.zeros(num_heads)
        sim = torch.zeros(num_heads, num_heads)
        # Pre-flatten once per head
        flattened = [h.reshape(-1, h.shape[-1]) for h in heads]
        for i in range(num_heads):
            mags[i] = torch.norm(heads[i], dim=-1).mean()
            for j in range(num_heads):
                sim[i, j] = F.cosine_similarity(flattened[i], flattened[j], dim=-1).mean()
        layer_similarities[layer_idx] = sim
        layer_magnitudes[layer_idx] = mags
    return layer_similarities, layer_magnitudes


# Compute similarities and magnitudes for CA and SA
ca_sims, ca_mags = compute_similarity_and_magnitudes(ca_head_outputs)
sa_sims, sa_mags = compute_similarity_and_magnitudes(sa_head_outputs)


# Derive groups and pivot scalings for all layers for a given sims/mags pair
def derive_pivots_for_all_layers(sims: dict[int, torch.Tensor], mags: dict[int, torch.Tensor], epsilon: float):
    layer_groups: dict[int, list[list[int]]] = {}
    layer_pivots: dict[int, list[int]] = {}
    layer_to_pivots: dict[int, list[tuple[int, float]]] = {}
    for layer_idx in range(num_layers):
        if layer_idx == 7:
            groups, pivot_scalings = pipeline.compute_groups_pivots_and_scalings(
                sims[layer_idx], mags[layer_idx], epsilon
            )
        else:
            groups, pivot_scalings = pipeline.compute_groups_pivots_and_scalings(sims[layer_idx], mags[layer_idx], 1)
        layer_groups[layer_idx] = groups
        layer_pivots[layer_idx] = [p for p, _ in pivot_scalings]
        layer_to_pivots[layer_idx] = pivot_scalings
    return layer_groups, layer_pivots, layer_to_pivots


ca_groups, ca_pivots, ca_layer_to_pivots = derive_pivots_for_all_layers(ca_sims, ca_mags, EPSILON)
sa_groups, sa_pivots, sa_layer_to_pivots = derive_pivots_for_all_layers(sa_sims, sa_mags, EPSILON)


# Display retention stats
def print_retention_stats(layer_to_pivots: dict[int, list[tuple[int, float]]], label: str):
    per_layer = {l: 100.0 * len(layer_to_pivots[l]) / num_heads for l in range(num_layers)}
    overall = 100.0 * sum(len(v) for v in layer_to_pivots.values()) / (num_layers * num_heads)
    print(f"\n[{label}] Percent retained per layer (%):")
    for l in range(num_layers):
        print(f"  Layer {l}: {per_layer[l]:.2f}%", "(" + str(len(layer_to_pivots[l])) + " heads)")
    print(
        f"[{label}] Overall percent retained across model: {overall:.2f}% ("
        + str(int(overall * num_layers * num_heads / 100))
        + " heads)"
    )


print_retention_stats(ca_layer_to_pivots, "CA")
print_retention_stats(sa_layer_to_pivots, "SA")

In [ ]:
layer_to_inspect = 7
print(f"\n[CA] Layer {layer_to_inspect} groups:")
pprint(ca_groups[layer_to_inspect])
print(f"[CA] Layer {layer_to_inspect} pivots: {ca_pivots[layer_to_inspect]}")
print(f"\n[SA] Layer {layer_to_inspect} groups:")
pprint(sa_groups[layer_to_inspect])
print(f"[SA] Layer {layer_to_inspect} pivots: {sa_pivots[layer_to_inspect]}")

In [ ]:
# Apply both CA and SA pivot-head manipulation hooks across all layers
pipeline.remove_pivot_head_manipulation_hooks(attention_type="ca")
pipeline.remove_pivot_head_manipulation_hooks(attention_type="sa")

pipeline.add_pivot_head_manipulation_hook(ca_layer_to_pivots, attention_type="ca")
pipeline.add_pivot_head_manipulation_hook(sa_layer_to_pivots, attention_type="sa")

print("Applied pivot-head manipulation across all layers for both CA and SA.")

### Prediction after CA + SA pivot-head manipulation
This runs a fresh prediction with both CA and SA pivot-head manipulations active and plots it alongside the original context.

In [ ]:
pred_after_both, _ = pipeline.predict_with_full_outputs(  # type: ignore
    context=context,
    prediction_length=prediction_length,
    num_samples=1,
    do_sample=False,
    use_cache=True,
    return_dict_in_generate=True,
    output_scores=False,
    limit_prediction_length=False,
)

import matplotlib.pyplot as plt
import numpy as np

context_np = context.squeeze().cpu().numpy()
pred_np = pred_after_both.squeeze().cpu().numpy()

context_timesteps = np.arange(len(context_np))
future_timesteps = np.arange(len(context_np), len(context_np) + len(pred_np))

fig, ax = plt.subplots(figsize=(6, 2))
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(future_timesteps, pred_np, color="tab:blue", linewidth=2, label="Prediction (CA+SA manip)")
ax.set_xlabel("Timestep", fontweight="bold")
ax.legend()
fig.tight_layout()
plt.show()